In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os
import gc
import time
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
# ==========================================
# 1. КОНФИГУРАЦИЯ И НАСТРОЙКИ
# ==========================================
DATA_ROOT = Path(r"/kaggle/input/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("/kaggle/working/models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 320
BATCH_SIZE = 32
EPOCHS =  100
N_SPLITS = 5
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

In [ ]:
# ==========================================
# 2. АУГМЕНТАЦИИ И ДАТАСЕТ
# ==========================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.2,
        scale_limit=0.2,
        rotate_limit=45,
        border_mode=cv2.BORDER_CONSTANT, fill=0,
        p=0.5),
    A.OneOf([
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.05,
            p=1.0),
        A.RandomGamma(gamma_limit=(80, 120), p=1.0), 
        ],
        p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(p=1.0),
        ],
        p=0.2),
    A.CoarseDropout(
        num_holes_range=(2, 6),
        hole_height_range=(16, 32),
        hole_width_range=(16, 32),
        fill=0,
        fill_mask=0,
        p=0.3
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class SegDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        mask_path = self.df.loc[idx, 'mask_path']

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask.unsqueeze(0)

In [ ]:
# ==========================================
# 3. ПОДГОТОВКА ДАННЫХ И СТРАТИФИКАЦИЯ
# ==========================================
def prepare_dataframe():
    """Собирает пути и считает площадь маски для стратификации объектов по размеру"""
    print("Подготовка датафрейма и расчет площадей масок...")
    data = []
    for mask_path in sorted(MASKS_DIR.glob("*.png")):
        img_path = IMAGES_DIR / (mask_path.stem + ".jpg")
        if img_path.exists():
            mask_preview = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            area = np.sum(mask_preview > 0)
            data.append({"image_path": str(img_path), "mask_path": str(mask_path), "area": area})
    
    df = pd.DataFrame(data)
    df['area_bin'] = pd.qcut(df['area'], q=5, labels=False)
    print(f"Найдено {len(df)} изображений.")
    return df

In [ ]:
# ==========================================
# 4. МЕТРИКИ И ФУНКЦИИ ОБУЧЕНИЯ (AMP)
# ==========================================
def dice_coef(y_pred, y_true, smooth=1e-6):
    y_pred = (torch.sigmoid(y_pred) > 0.5).float()
    intersection = (y_pred * y_true).sum(dim=(2, 3))
    union = y_pred.sum(dim=(2, 3)) + y_true.sum(dim=(2, 3))
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean().item()

def train_one_fold(model, train_loader, val_loader, optimizer, scheduler, criterion, fold, conf_name, epochs=EPOCHS):
    scaler = torch.amp.GradScaler('cuda')
    best_dice = 0.0
    best_epoch = 0
    history = {'train_loss': [], 'val_loss': [], 'val_dice': []}
    
    save_path = SAVE_DIR / f"{conf_name}_fold{fold}.pth"

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        
        # ================= TRAIN =================
        model.train()
        train_loss = 0.0
        
        for images, masks in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            scheduler.step() 
            
        train_loss /= len(train_loader)

        # VALIDATION
        model.eval()
        val_loss, val_dice = 0.0, 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, masks)
                    
                val_loss += loss.item()
                val_dice += dice_coef(logits, masks)
                
        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        # ЛОГИРОВАНИЕ
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        
        epoch_time = time.time() - start_time

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            torch.save(model.state_dict(), save_path)
            print(f"   Epoch {epoch:02d}/{EPOCHS} [{epoch_time:.0f}s] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} -> Новый лучший Val Dice")
        else:
            print(f"   Epoch {epoch:02d}/{EPOCHS} [{epoch_time:.0f}s] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")
            
    print(f"✅ Fold {fold} завершен. Лучший Dice: {best_dice:.4f} на эпохе {best_epoch}. Сохранено: {save_path.name}")
    return history, best_dice, best_epoch

def plot_history(histories, conf_name):
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    for i, hist in enumerate(histories):
        plt.plot(hist['train_loss'], label=f'Train Fold {i+1}', linestyle='--')
        plt.plot(hist['val_loss'], label=f'Val Fold {i+1}')
    plt.title(f"{conf_name}: Loss")
    plt.legend(fontsize=8)
    
    plt.subplot(1, 2, 2)
    for i, hist in enumerate(histories):
        plt.plot(hist['val_dice'], label=f'Dice Fold {i+1}')
    plt.title(f"{conf_name}: Validation Dice")
    plt.legend(fontsize=8)
    
    plt.show()

In [ ]:
# ==========================================
# 5. ГЛАВНЫЙ ЦИКЛ K-FOLD ДЛЯ ОДНОЙ КОНФИГУРАЦИИ
# ==========================================
def run_experiment(df, conf_name, arch, encoder, lr, weight_decay, epochs=EPOCHS):
    print(f"\n{'='*50}\n🚀 ЗАПУСК: {conf_name}\nАрхитектура: {arch} | Энкодер: {encoder}")
    print(f"Параметры: LR={lr}, Weight Decay={weight_decay}\n{'='*50}")
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    
    criterion = smp.losses.DiceLoss(mode='binary', from_logits=True)
    bce = nn.BCEWithLogitsLoss()
    combo_loss = lambda y_pred, y_true: 0.5 * criterion(y_pred, y_true) + 0.5 * bce(y_pred, y_true)

    histories = []
    best_dices = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['area_bin']), 1):
        print(f"\n--- Обучение Fold {fold}/{N_SPLITS} ---")
        
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        train_loader = DataLoader(SegDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(SegDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        
        if arch == "UnetPlusPlus":
            model = smp.UnetPlusPlus(encoder_name=encoder, encoder_weights="imagenet", in_channels=3, classes=1)
        elif arch == "MAnet":
            model = smp.MAnet(encoder_name=encoder, encoder_weights="imagenet", in_channels=3, classes=1)
        elif arch == "DeepLabV3Plus":
            model = smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights="imagenet", in_channels=3, classes=1)
            
        model.to(DEVICE)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        steps_per_epoch = len(train_loader)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=lr,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            pct_start=0.15,
            div_factor=25.0,
            final_div_factor=1000.0
        )
        
        history, b_dice, b_epoch = train_one_fold(model, train_loader, val_loader, optimizer, scheduler, combo_loss, fold, conf_name)
        
        histories.append(history)
        best_dices.append(b_dice)
        
        del model, optimizer, scheduler, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()
        
    print(f"\n🏆 Итоги {conf_name}: Средний Dice по всем фолдам: {np.mean(best_dices):.4f}")
    plot_history(histories, conf_name)

In [ ]:
# ==========================================
# 6. ЗАПУСК КОНФИГУРАЦИИ 2
# ==========================================
df = prepare_dataframe()

# Конфигурация 2: MAnet mit_b3
run_experiment(
        df, conf_name="conf2_manet_mitb3", arch="MAnet", encoder="mit_b3",
        lr=2e-4, weight_decay=1e-2
    )